# Kronos-NSE — evaluation grid on Kaggle

> Research/education tool — scenario visualization, not investment advice.

Runs the A3 evaluation grid on a Kaggle T4/P100 (16 GB), which finishes in roughly 1–2 hours
against 4–6 on a 6 GB laptop — and without the GPU dropping off the bus.

## Before you run anything

In the notebook sidebar:

1. **Accelerator → GPU T4 x2** (or P100). Without this the grid will not finish.
2. **Internet → On.** Needed to clone the repo and download the model weights.
3. **Attach the corpus dataset** — see the next cell.

## Getting `data/` here

`data/` is gitignored, so it does not arrive with the clone. On your local machine, zip
`data/` (17.6 MB) and upload it as a **Kaggle Dataset** (Datasets → New Dataset), then
attach it to this notebook. It will appear under `/kaggle/input/<your-dataset-name>/`.

**Do not re-run `fetch_nse.py` here.** Canonical prices are split/bonus back-adjusted, and
back-adjustment rewrites history — a corpus fetched on a different date is a *different
corpus*. Nothing would error; the numbers would simply stop being comparable to every
measurement taken so far.

## A note on exact reproduction

The baseline numbers in the verification cell are pure NumPy and **must** match exactly —
they prove the corpus and harness survived the move. The Kronos ensembles may differ in
the last digits from a different GPU architecture, which is normal. Do not mix devices
*within* one grid; finish a run on the hardware it started on, or resume on the same kind.

In [ ]:
import glob
import os
import pathlib
import shutil
import subprocess
import sys

# ---- configure -------------------------------------------------------------
REPO_URL = "https://github.com/neopentane7/kronos-candlecast.git"
DATA_DIR = None  # e.g. "/kaggle/input/kronos-nse-corpus"; None = autodetect

# Private repo? Add a GitHub token as a Kaggle Secret named GITHUB_TOKEN
# (Add-ons -> Secrets). Leave this alone if the repo is public.
USE_TOKEN = True

BATCH_SIZE = 24  # 16 GB card; drop to 12 if you hit OOM
CHECKPOINT_EVERY = 5  # flush partials every N batches
SPLIT = "test"
LIMIT = None  # e.g. 60 for a quick smoke run; None = full 708-window grid

WORK = pathlib.Path("/kaggle/working")
REPO = WORK / "kronos-candlecast"
print("working dir:", WORK)

In [ ]:
# ---- 1. clone the project and the pinned upstream --------------------------
UPSTREAM_SHA = "67b630e67f6a18c9e9be918d9b4337c960db1e9a"
UPSTREAM_URL = "https://github.com/shiyu-coder/Kronos.git"


def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True).stdout.strip()


url = REPO_URL
if USE_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
        print("using GITHUB_TOKEN from Kaggle Secrets")
    except Exception as exc:  # noqa: BLE001 - fall back to an anonymous clone
        print(f"no usable token ({type(exc).__name__}); trying anonymous clone")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True)
print("repo:", git("-C", str(REPO), "log", "--oneline", "-1"))

# Upstream is gitignored by design and reproduced at its pinned commit. The harness
# overlays its generation loop and an equivalence test asserts bit-identical output,
# so the SHA is not optional.
up = REPO / "phase-a" / "Kronos"
if not up.exists():
    subprocess.run(["git", "clone", "--quiet", UPSTREAM_URL, str(up)], check=True)
    subprocess.run(
        ["git", "-C", str(up), "checkout", "--quiet", "--detach", UPSTREAM_SHA], check=True
    )
print("upstream:", git("-C", str(up), "rev-parse", "HEAD")[:12])

In [ ]:
# ---- 2. dependencies -------------------------------------------------------
# Kaggle already ships torch with a working CUDA build, so we install only what is
# missing rather than re-resolving the lockfile (which would pull a ~2.5 GB torch
# wheel). The trade-off: exact package pinning is relaxed. The guard is the
# verification cell below, which re-derives the baseline numbers from NumPy alone,
# so a broken environment cannot pass silently.
!pip install -q scoringrules pandera duckdb exchange_calendars einops 2>&1 | tail -2

import torch  # noqa: E402 - installed by the line above

print(f"torch {torch.__version__} | cuda {torch.version.cuda}")
print(f"cuda available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Set Accelerator -> GPU T4 x2 in the sidebar.")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory / 2**30:.1f} GiB")

In [ ]:
# ---- 3. restore the corpus -------------------------------------------------
src = DATA_DIR
if src is None:
    hits = glob.glob("/kaggle/input/*/parquet") + glob.glob("/kaggle/input/*/data/parquet")
    if not hits:
        raise SystemExit(
            "No corpus found under /kaggle/input. Upload data/ as a Kaggle Dataset "
            "and attach it, or set DATA_DIR explicitly."
        )
    src = str(pathlib.Path(hits[0]).parent)
    print("autodetected corpus at", src)

dest = REPO / "data"
dest.mkdir(exist_ok=True)
for item in pathlib.Path(src).iterdir():
    target = dest / item.name
    if target.exists():
        continue
    (shutil.copytree if item.is_dir() else shutil.copy2)(item, target)

n_parts = len(list((dest / "parquet").glob("*/*.parquet")))
print(f"corpus restored: {n_parts} ticker partitions")
assert n_parts > 0, "no parquet partitions found — check the dataset layout"

In [ ]:
# ---- 4. VERIFY THE PORT ----------------------------------------------------
# Baselines are deterministic from the grid and the seed, so these four numbers must
# match exactly. If they do, the corpus, window enumeration, metric layer and seeding
# all survived the move. If they differ, the corpus is not the same one — almost
# certainly because fetch_nse.py was re-run somewhere.
os.chdir(REPO)
!python phase-a/eval/calibrate.py --split test --skip-model --no-figures 2>&1 | tail -12

print("""
EXPECTED
  last_value          CRPS 89.0381   cov@80 0.0008   IS 890.381
  random_walk_drift   CRPS 67.2363   cov@80 0.8369   IS 462.799
  effective blocks: 22

Stop here if these do not match.
""")

In [ ]:
# ---- 5. run the grid -------------------------------------------------------
# Resumable: if a previous session left a run directory with a partial, point --resume
# at it and only the missing batches are recomputed. Resume is bit-identical, because
# each batch is seeded from its own offset.
RESUME_DIR = None  # e.g. "results/20260804T...."; None = fresh run

cmd = [
    sys.executable,
    "phase-a/eval/calibrate.py",
    "--split",
    SPLIT,
    "--batch-size",
    str(BATCH_SIZE),
    "--checkpoint-every",
    str(CHECKPOINT_EVERY),
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if RESUME_DIR:
    cmd += ["--resume", RESUME_DIR]

print(" ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
print("exit code:", proc.wait())

In [ ]:
# ---- 6. offline analysis (CPU only) ----------------------------------------
# subprocess rather than a `!` escape: shell escapes inside an if-block are fragile
# and do not survive tooling that parses the notebook as Python.
runs = sorted(glob.glob("results/*/ensembles.npz"))
if not runs:
    print("no ensembles.npz yet — run the grid cell first")
else:
    latest = str(pathlib.Path(runs[-1]).parent)
    print("analysing", latest, flush=True)
    r = subprocess.run(
        [sys.executable, "phase-a/eval/run_analysis.py", latest],
        capture_output=True,
        text=True,
    )
    print(r.stdout or r.stderr)

In [ ]:
# ---- 7. package results for download ---------------------------------------
# Only /kaggle/working survives the session. Zip results/ so it can be downloaded
# from the notebook's Output tab, then copied back into the local repo.
out = "/kaggle/working/kronos_results"
shutil.make_archive(out, "zip", root_dir=str(REPO), base_dir="results")
size = os.path.getsize(out + ".zip") / 2**20
print(f"{out}.zip  ({size:.1f} MB)")
print("\nDownload from the Output panel, unzip into the local repo, then:")
print("  uv run python phase-a/eval/run_analysis.py results/<run-dir>")

## If the session ends mid-run

Nothing is lost. Partials flush every `CHECKPOINT_EVERY` batches.

1. Download `kronos_results.zip` from the Output panel before the session expires.
2. In the next session, re-run cells 1–4, upload the zip as a dataset (or re-clone and
   restore `results/`), set `RESUME_DIR` to the run directory, and re-run cell 5.

Only the missing batches are recomputed, and the result is identical to an uninterrupted
run — `tests/test_resume.py` asserts that at `rtol=0, atol=0`.

## Tuning

| symptom | change |
|---|---|
| CUDA out of memory | `BATCH_SIZE` 24 → 12 → 6 |
| Want a quick smoke test first | `LIMIT = 60` (~5 min) |
| Session keeps dying | lower `CHECKPOINT_EVERY` to 2 |

Peak VRAM was 5.9 GB at batch 6 on a 6 GB card, so batch 24 on a 16 GB card should sit
comfortably inside budget.